# 1.Import libraries

In [36]:
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sqlalchemy import create_engine



# connect to Workbench

In [37]:
username = 'root'
password = '08042004'
host = 'localhost'
port = '3306'
database = 'law_db'

# Tạo engine kết nối đến MySQL
engine = create_engine(
    f"mysql+mysqlconnector://{username}:{password}@{host}:{port}/{database}?charset=utf8mb4"
)


# Đọc dữ liệu

In [122]:
# Lấy dữ liệu
df_case = pd.read_sql("SELECT id, content, text FROM `case` LIMIT 5000", con = engine)
df_law = pd.read_sql("SELECT id, article, title, content, type FROM law", con = engine)
df_case_law = pd.read_sql("SELECT case_id, law_id FROM case_law", con = engine)

In [39]:
df_case.head()


,id,content
0,1,Vũ Tuấn T về tội “Cho vay lãi nặng trong giao ...
1,2,Tuyên bố bị cáo Trần Anh D phạm tội “Vi phạm ...
2,3,Cao Văn L
3,4,"Khoảng 12 giờ 30 phút ngày 26/01/2018, Phan Th..."
4,5,Vi Van T phạm tội vi phạm quy định về quản lý...


In [124]:
df_law.head()


,id,article,title,content,type
0,1,Điều 1,Nhiệm vụ của Bộ luật hình sự,"""{\""info\"": \""Bộ luật hình sự có nhiệm vụ bảo vệ chủ quyền quốc gia, an ninh của đất nước, bảo vệ chế độ xã hội chủ nghĩa, quyền con người, quyền công dân, bảo vệ quyền bình đẳng giữa đồng bào các dân tộc, bảo vệ lợi ích của Nhà nước, tổ chức, bảo vệ trật tự pháp luật, chống mọi hành vi phạm tội; giáo dục mọi người ý thức tuân theo pháp luật, phòng ngừa và đấu tranh chống tội phạm. Bộ luật này quy định về tội phạm và hình phạt.\""}""",hình sự
1,2,Điều 2,Cơ sở của trách nhiệm hình sự,"""{\""info\"": \""\"", \""1\"": \""Chỉ người nào phạm một tội đã được Bộ luật hình sự quy định mới phải chịu trách nhiệm hình sự.\"", \""2\"": \""Chỉ pháp nhân thương mại nào phạm một tội đã được quy định tại Điều 76 của Bộ luật này mới phải chịu trách nhiệm hình sự.\""}""",hình sự
2,3,Điều 3,Nguyên tắc xử lý,"""{\""info\"": \""\"", \""1\"": {\""a\"": \""Mọi hành vi phạm tội do người thực hiện phải được phát hiện kịp thời, xử lý nhanh chóng, công minh theo đúng pháp luật;\"", \""b\"": \""Mọi người phạm tội đều bình đẳng trước pháp luật, không phân biệt giới tính, dân tộc, tín ngưỡng, tôn giáo, thành phần, địa vị xã hội;\"", \""c\"": \""Nghiêm trị người chủ mưu, cầm đầu, chỉ huy, ngoan cố chống đối, côn đồ, tái phạm nguy hiểm, lợi dụng chức vụ, quyền hạn để phạm tội;\"", \""d\"": \""Nghiêm trị người phạm tội dùng thủ đo...",hình sự
3,4,Điều 4,Trách nhiệm phòng ngừa và đấu tranh chống tội phạm,"""{\""info\"": \""\"", \""1\"": \""Cơ quan Công an, Viện kiểm sát nhân dân, Tòa án nhân dân và các cơ quan hữu quan khác có trách nhiệm thực hiện đầy đủ chức năng, nhiệm vụ, quyền hạn của mình, đồng thời hướng dẫn, giúp đỡ các cơ quan khác của Nhà nước, tổ chức, cá nhân phòng ngừa và đấu tranh chống tội phạm, giám sát và giáo dục người phạm tội tại cộng đồng.\"", \""2\"": \""Cơ quan, tổ chức có nhiệm vụ giáo dục những người thuộc quyền quản lý của mình nâng cao cảnh giác, ý thức bảo vệ và tuân theo pháp...",hình sự
4,5,Điều 5,Hiệu lực của Bộ luật hình sự đối với những hành vi phạm tội trên lãnh thổ nước Cộng hòa xã hội chủ nghĩa Việt Nam,"""{\""info\"": \""\"", \""1\"": \""Bộ luật hình sự được áp dụng đối với mọi hành vi phạm tội thực hiện trên lãnh thổ nước Cộng hòa xã hội chủ nghĩa Việt Nam. Quy định này cũng được áp dụng đối với hành vi phạm tội hoặc hậu quả của hành vi phạm tội xảy ra trên tàu bay, tàu biển mang quốc tịch Việt Nam hoặc tại vùng đặc quyền kinh tế, thềm lục địa của Việt Nam.\"", \""2\"": \""Đối với người nước ngoài phạm tội trên lãnh thổ nước Cộng hòa xã hội chủ nghĩa Việt Nam thuộc đối tượng được hưởng quyền miễn trừ ...",hình sự


In [41]:
df_case_law.head()

,case_id,law_id
0,1,None
1,1,None
2,1,None
3,1,None
4,1,None


In [72]:
from underthesea import word_tokenize
import unicodedata
import re

stopwords = set(["và", "là", "có", "cho", "của", "các", "những", "khi", "thì", "được", "từ", "rằng", 
    "với", "một", "như", "đã", "này", "trong", "trên", "đến", "đi", "về", "ở", "bị", "do"])

def preprocess_text(text):
    # text k la str => rỗng
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFC", text) # chuẩn hóa mã hóa Unicode
    text = re.sub(r"[^a-zA-ZÀ-Ỹà-ỹ0-9\s]", " ", text) #giữ chữ, số , loại kí tự đbiet
    text = re.sub(r"\s+", " ", text).strip()        # bỏ khoảng trắng thừa
    text = text.lower()
    words = word_tokenize(text, format="text").split() #tách từ = unicode data
    
    filtered_words = [] # loc stop word
    for w in words:
        if w not in stopwords:
            filtered_words.append(w)
    return " ".join(words)




# TIỀN XỬ LÝ BẢNG CASE

In [49]:
df_case['clean_text'] = df_case['text'].apply(preprocess_text)


In [50]:
# In 5 dòng đầu để kiểm tra
df_case[['text', 'clean_text']].head()


,text,clean_text
0,...,tòa_án nhân_dân quận ngô_quyền thành_phố hải_p...
1,1 \n \nTÒA ÁN NHÂN DÂN \nCỘNG HÒA XÃ HỘI CHỦ N...,1 tòa_án nhân_dân cộng_hòa xã_hội_chủ_nghĩa_vi...
2,\n1 \nTÒA ÁN NHÂN DÂN QUẬN \nLONG BIÊN – TP H...,1 tòa_án nhân_dân quận long_biên tp hà_nội bản...
3,TÒA ÁN NHÂN DÂN \nCỘNG HÒA XÃ HỘI CHỦ N...,tòa_án nhân_dân cộng_hòa xã_hội_chủ_nghĩa_việt...
4,2 \nTÒA ÁN NHÂN DÂN \nHUYỆN QUAN SƠN \nTỈNH TH...,2 tòa_án nhân_dân huyện quan_sơn tỉnh thanh_hó...


# BẢNG LAW

In [143]:
import json
import pandas as pd

def expand_law_json(df_law):
    rows = []

    for _, row in df_law.iterrows():
        law_id = row["id"]
        article = row["article"]
        title = row["title"]
        law_type = row["type"]

        try:
            parsed = row["content"]
            if isinstance(parsed, str):
                parsed = json.loads(parsed)   # lần 1
                if isinstance(parsed, str): parsed = json.loads(parsed)  # lần 2 (nếu vẫn là chuỗi)
            if not isinstance(parsed, dict):
                print(f"k là dict tại id={law_id}")
                continue
        except json.JSONDecodeError:
            print(f"lỗi json decode tại ={law_id}: {e}")
            continue

        for clause_key, clause_value in parsed.items():
            if clause_key == "info":
                # đoạn mô tả chung
                if not clause_value or clause_value.strip() == "":
                    continue  # bỏ qua nếu info rỗng
                rows.append({
                    "id": law_id,
                    "article": article,
                    "clause": None,
                    "point": None,
                    "text": clause_value.strip(),
                    "title": title,
                    "type": law_type
                })
            elif isinstance(clause_value, str):
                rows.append({
                    "id": law_id,
                    "article": article,
                    "clause": clause_key,
                    "point": None,
                    "text": clause_value.strip(),
                    "title": title,
                    "type": law_type
                })
            elif isinstance(clause_value, dict):
                for point_key, point_text in clause_value.items():
                    rows.append({
                        "id": law_id,
                        "article": article,
                        "clause": clause_key,
                        "point": point_key,
                        "text": point_text.strip(),
                        "title": title,
                        "type": law_type
                    })

    return pd.DataFrame(rows)


In [144]:
df_expanded_law = expand_law_json(df_law)
df_expanded_law = df_expanded_law[df_expanded_law["text"].notnull()]
df_expanded_law.head()


,id,article,clause,point,text,title,type
0,1,Điều 1,None,None,"Bộ luật hình sự có nhiệm vụ bảo vệ chủ quyền quốc gia, an ninh của đất nước, bảo vệ chế độ xã hội chủ nghĩa, quyền con người, quyền công dân, bảo vệ quyền bình đẳng giữa đồng bào các dân tộc, bảo vệ lợi ích của Nhà nước, tổ chức, bảo vệ trật tự pháp luật, chống mọi hành vi phạm tội; giáo dục mọi người ý thức tuân theo pháp luật, phòng ngừa và đấu tranh chống tội phạm. Bộ luật này quy định về tội phạm và hình phạt.",Nhiệm vụ của Bộ luật hình sự,hình sự
1,2,Điều 2,1,None,Chỉ người nào phạm một tội đã được Bộ luật hình sự quy định mới phải chịu trách nhiệm hình sự.,Cơ sở của trách nhiệm hình sự,hình sự
2,2,Điều 2,2,None,Chỉ pháp nhân thương mại nào phạm một tội đã được quy định tại Điều 76 của Bộ luật này mới phải chịu trách nhiệm hình sự.,Cơ sở của trách nhiệm hình sự,hình sự
3,3,Điều 3,1,a,"Mọi hành vi phạm tội do người thực hiện phải được phát hiện kịp thời, xử lý nhanh chóng, công minh theo đúng pháp luật;",Nguyên tắc xử lý,hình sự
4,3,Điều 3,1,b,"Mọi người phạm tội đều bình đẳng trước pháp luật, không phân biệt giới tính, dân tộc, tín ngưỡng, tôn giáo, thành phần, địa vị xã hội;",Nguyên tắc xử lý,hình sự
